In [1]:
!apt-get install -y zstd

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 42 not upgraded.
Need to get 603 kB of archives.
After this operation, 1,695 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 zstd amd64 1.4.8+dfsg-3build1 [603 kB]
Fetched 603 kB in 5s (133 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../zstd_1.4.8+dfsg-3build1_amd64.deb ...
Unpacking zstd (1.4.8+dfsg-3build1) ...
Setting up zstd (1.4.8+dfsg-3build1) ...
Processing triggers for man-db (2.10.2-1) ...


In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
import subprocess
import time

subprocess.Popen(["ollama", "serve"])
time.sleep(3)
print("Ollama başlatıldı!")

Ollama başlatıldı!


In [4]:
!ollama pull llama3.1:latest


In [5]:
from datasets import load_dataset
meetingbank = load_dataset("huuuyeah/meetingbank")

train_data = meetingbank['train']
test_data = meetingbank['test']
val_data = meetingbank['validation']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.json:   0%|          | 0.00/88.4M [00:00<?, ?B/s]

validation.json:   0%|          | 0.00/13.2M [00:00<?, ?B/s]

test.json:   0%|          | 0.00/13.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5169 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/861 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/862 [00:00<?, ? examples/s]

In [6]:
def generator(data_split):
  for instance in data_split:
    yield instance['id'], instance['summary'], instance['transcript']

In [ ]:
import requests
import json
import csv
import time

SYSTEM_PROMPT = """You are a meeting assistant. Given a meeting transcript, produce a JSON object with two keys:
1. "summary": A concise, well-structured meeting summary in paragraph form. Cover the key discussion points, decisions made, and conclusions.
2. "action_items": An array of action items extracted from the meeting. Each item is an object with:
   - "description": A clear, actionable description of what needs to be done.
   - "assignee": The name of the person responsible. ONLY use names that actually appear as speakers in the transcript. Set to null if no specific person was assigned.
IMPORTANT: Do NOT invent or hallucinate assignee names. Only use speaker names that appear in the transcript.
Return ONLY valid JSON, no markdown fences, no extra text. Example format:
{"summary": "The team discussed...", "action_items": [{"description": "Task description here", "assignee": null}]}"""


def generate_summary(transcript):
    prompt = f"{SYSTEM_PROMPT}\n\nTranscript:\n{transcript}"

    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "llama3.1",
            "prompt": prompt,
            "stream": False
        }
    )

    raw = response.json()["response"].strip()

    try:
        parsed = json.loads(raw)
        return parsed
    except json.JSONDecodeError:
        return raw


output_file = "summaries.csv"

with open(output_file, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["id", "transcript", "summary", "generated_response"])
    writer.writeheader()

    for id_val, summary, transcript in generator(train_data):
        if id_val == 100:
            break

        print(f"Processing ID: {id_val}...")

        generated_response = generate_summary(transcript)

        writer.writerow({
            "id": id_val,
            "transcript": transcript,
            "summary": summary,
            "generated_response": json.dumps(generated_response, ensure_ascii=False)
        })

        time.sleep(0.5)

print(f"Tamamlandı! '{output_file}' dosyasına kaydedildi.")

Processing ID: 0...


ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/generate (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x78370d0e41a0>: Failed to establish a new connection: [Errno 111] Connection refused'))

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
!pip install deepeval

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 844.6/844.6 kB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.9/226.9 kB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 98.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 4.7 MB/s eta 0:00:00


In [9]:
!pip install ollama

In [10]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
import pandas as pd
import json
from deepeval.models import OllamaModel



In [11]:
import os
import time
import json
import pandas as pd
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.models import OllamaModel

In [ ]:
import os
import time
import json
import pandas as pd
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric
from deepeval.models import OllamaModel

os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"

df = pd.read_csv("summaries.csv")

MEETING_ASSESSMENT_QUESTIONS = [
    "Does the summary contain any information not present in the transcript?",
    "Does the summary contradict anything stated in the transcript?",
    "Are the main topics of the meeting captured in the summary?",
    "If any action items or next steps were discussed, are they reflected in the summary?",
]

metric = SummarizationMetric(
    threshold=0.3,
    model=OllamaModel(model="llama3.1"),
    verbose_mode=True,
    assessment_questions=MEETING_ASSESSMENT_QUESTIONS,
    n=0
)

results = []

for i, row in df.head(100).iterrows():
    # Summary parse
    try:
        parsed_content = json.loads(row["generated_response"])
        if isinstance(parsed_content, dict):
            generated_summary = parsed_content.get("summary", str(parsed_content))
        else:
            generated_summary = str(parsed_content)
    except (json.JSONDecodeError, TypeError):
        generated_summary = str(row["generated_response"])

    test_case = LLMTestCase(
        input=row["transcript"],
        actual_output=generated_summary,
    )

    try:
        metric.measure(test_case)
        results.append({
            "index": i,
            "score": metric.score,
            "reason": metric.reason,
            "passed": metric.is_successful()
        })
        print(f"[{i+1}/10] Score: {metric.score:.2f} | {metric.reason}")
    except Exception as e:
        results.append({
            "index": i,
            "score": None,
            "reason": str(e),
            "passed": False
        })
        print(f"[{i+1}/10] HATA: {e}")

    time.sleep(3)  # Ollama için kısa bir bekleme yeterli

# Sonuçları kaydet
results_df = pd.DataFrame(results)
results_df.to_csv("evaluation_results.csv", index=False)
print(f"\nOrtalama Skor: {results_df['score'].mean():.2f}")
print(f"Başarılı: {results_df['passed'].sum()}/10")

In [12]:
!pip install rouge-score


  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=049665d78f97f090fec0c1274b554b0fef2a08ce152c5a640205417e5c2db417
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [14]:
import os
import time
import json
import pandas as pd
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric
from deepeval.models import OllamaModel

os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"

df = pd.read_csv("/summaries.csv")

metric = FaithfulnessMetric(
    threshold=0.7,
    model=OllamaModel(model="llama3.1"),
    verbose_mode=True
)

faithfulness_results = []

for i, row in df.head(100).iterrows():
    try:
        parsed_content = json.loads(row["generated_response"])
        if isinstance(parsed_content, dict):
            generated_summary = parsed_content.get("summary", str(parsed_content))
        else:
            generated_summary = str(parsed_content)
    except (json.JSONDecodeError, TypeError):
        generated_summary = str(row["generated_response"])

    test_case = LLMTestCase(
        input=row["transcript"],
        actual_output=generated_summary,
        retrieval_context=[row["transcript"]]  # FaithfulnessMetric için zorunlu
    )

    try:
        metric.measure(test_case)
        faithfulness_results.append({
            "index": i,
            "score": metric.score,
            "reason": metric.reason,
            "passed": metric.is_successful()
        })
        print(f"[{i+1}/100] Score: {metric.score:.2f} | {metric.reason}")
    except Exception as e:
        faithfulness_results.append({"index": i, "score": None, "reason": str(e), "passed": False})
        print(f"[{i+1}/100] HATA: {e}")

    time.sleep(3)

faithfulness_df = pd.DataFrame(faithfulness_results)
faithfulness_df.to_csv("faithfulness_results.csv", index=False)
print(f"\nOrtalama Skor: {faithfulness_df['score'].mean():.2f}")
print(f"Başarılı: {faithfulness_df['passed'].sum()}/100")

Output()

**************************************************

Faithfulness Verbose Logs

**************************************************

Truths (limit=None):
[
    "Council Bill 161 has passed as amended",
    "Nine yeses, two nays",
    "The moratorium is up on May 26"
] 
 
Claims:
[
    "Council Bill 161 proposed a moratorium on parking permits for micro-units.",
    "Micro-units typically have limited parking spaces.",
    "The bill would help address concerns about parking in neighborhoods where micro-units were being built.",
    "Councilman Clarke presented the bill and explained its purpose.",
    "Councilman Espinosa thanked Councilman Brooks for leading the moratorium process.",
    "This is a complex issue that requires input from various stakeholders.",
    "The Colfax Viaduct was originally built for transit and not automobiles.",
    "It's time to prioritize pedestrian-friendly infrastructure in Denver.",
    "Councilwoman Ortega expressed concerns about the quality of design for micro-units.",
    "Micro-units should be designed to last for generations rather than needing frequent replacements.",
    "The Colfax Viaduct was built in 1916 for Denver Tramway at a cost of $715,000.",
    "Councilwoman Robb originally took up this issue in 2011.",
    "Neighbors have been very engaged in the process.",
    "This is a complex issue that requires careful consideration.",
    "The council voted on the bill with a final tally of 9-2 in favor of passing the amended bill.",
    "Councilwoman Black did not vote."
] 
 
Verdicts:
[
    {
        "verdict": "no",
        "reason": "The claim states that Council Bill 161 proposed a moratorium on parking permits for micro-units,
but the retrieval context indicates that the bill has passed as amended and there is no mention of proposing a 
moratorium."
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "no",
        "reason": "The claim states that Councilman Clarke presented the bill, but the retrieval context does not 
mention this specific detail."
    },
    {
        "verdict": "idk",
        "reason": "There is no information in the retrieval context about Councilman Espinosa thanking Councilman 
Brooks for leading the moratorium process."
    },
    {
        "verdict": "no",
        "reason": "The claim states that this is a complex issue, but the retrieval context does not indicate any 
complexity or controversy surrounding the bill."
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "idk",
        "reason": "There is no information in the retrieval context about prioritizing pedestrian-friendly 
infrastructure in Denver."
    },
    {
        "verdict": "no",
        "reason": "The claim states that Councilwoman Ortega expressed concerns, but the retrieval context does not
mention any specific concerns from her."
    },
    {
        "verdict": "idk",
        "reason": "There is no information in the retrieval context about micro-units being designed to last for 
generations rather than needing frequent replacements."
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "no",
        "reason": "The claim states that Councilwoman Robb originally took up this issue, but the retrieval context
does not mention her involvement in 2011 or any other year."
    },
    {
        "verdict": "idk",
        "reason": "There is no information in the retrieval context about neighbors being very engaged in the 
process."
    },
    {
        "verdict": "no",
        "reason": "The claim states that this is a complex issue, but the retrieval context does not indicate any 
complexity or controversy surrounding the bill."
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "idk",
        "reason": "There is no information in the retrieval context about Councilwoman Black's vote."
    }
]
 
Score: 0.625
Reason: The score is 0.62 because the actual output contains several inaccuracies and omissio

...

Faithfulness Verbose Logs

**************************************************

Truths (limit=None):
[
    "Council Bill 400 was a six-month extension for the photo red light program.",
    "The Denver Post editorial staff suggested modifying the photo red light program.",
    "Councilwoman Fox voted against Council Bill 400.",
    "Councilman Kenney voted in favor of Council Bill 400, but stated it would be his last vote unless significant 
changes were made.",
    "Council Bill 371 was a proposal to change the contracting process from two readings by ordinance to one 
reading by contract.",
    "The proposed change in Council Bill 371 was intended to save $39,000 in the city's $2 billion budget.",
    "Councilwoman Ortega voted against Council Bill 371.",
    "Councilman Lopez voted against Council Bill 371.",
    "Councilwoman Shepherd voted against Council Bill 371.",
    "Councilman Montero voted against Council Bill 371.",
    "Councilman Fights voted against Council Bill 371.",
    "Council Bill 512 was an ordinance dealing with health insurance for employees and allowing high deductible 
health plans with accompanying health savings account contributions.",
    "Federal law prohibits some individuals from having a health savings account, which is why a subsequent 
ordinance would be needed to ensure equity among employees.",
    "The resolutions on the floor for adoption were Resolutions 34, 36, 37, 39, 68, 469, 91, and 90."
] 
 
Claims:
[
    "Council Bill 356 was passed on final consideration after a vote of 8-4 with one abstention.",
    "The bill deals with vacating a portion of an alley to allow for a proposed development on a slope.",
    "Councilwoman Ortega questioned the decision, suggesting that the vacation should be divided among all adjacent
property owners.",
    "Councilman Lopez spoke against the bill, arguing that it weakens the legislative branch's authority and gives 
too much power to staff.",
    "The council also voted on Council Bill 400, which extends the photo red light program for another six 
months.",
    "Council Bill 400 was passed with a vote of 12-1, with one nay and one abstention.",
    "Council Bill 371, which aims to change the contracting process to allow for faster approval of contracts, was 
discussed but ultimately failed to pass after a vote of 8-4 with four nays and one abstention.",
    "The council voted on Council Bill 400, which extends the photo red light program for another six months.",
    "Council Bill 371 aims to change the contracting process to allow for faster approval of contracts."
] 
 
Verdicts:
[
    {
        "verdict": "no",
        "reason": "The claim states Council Bill 356 was passed with a vote of 8-4, but the context does not 
mention this bill."
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "idk",
        "reason": "Councilwoman Ortega questioned the decision, suggesting that the vacation should be divided 
among all adjacent property owners. However, it is unclear if she voted against or for the bill."
    },
    {
        "verdict": "no",
        "reason": "The claim states Councilman Lopez spoke against the bill, arguing that it weakens the 
legislative branch's authority and gives too much power to staff. However, this is not mentioned in the context."
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "no",
        "reason": "The claim states Council Bill 400 was passed with a vote of 12-1, but the context states it was 
a six-month extension for the photo red light program and was voted on by Councilwoman Fox against."
    },
    {
        "verdict": "yes",
        "reason": null
    },
    {
        "verdict": "no",
        "reason": "The claim states Council Bill 371 aims to change the contracting process to allow for faster 
approval of contracts, but the context states it was a proposal to change the contracting process from two readings
by ordinance to one reading by contract."
    }
]
 
Score: 0.5
Reason: The score is 0.50 becaus

======================================================================

[100/100] Score: 0.50 | The score is 0.50 because there are multiple contradictions in the actual output, including incorrect information about Council Bill 356 and 400's votes, and a mischaracterization of Councilman Lopez's stance on the bill, as well as an error in describing Council Bill 371's purpose.

Ortalama Skor: 0.62
Başarılı: 19/100


In [16]:
import os
import time
import json
import pandas as pd
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.metrics import GEval
from deepeval.models import OllamaModel

os.environ["DEEPEVAL_PER_ATTEMPT_TIMEOUT_SECONDS_OVERRIDE"] = "300"

df = pd.read_csv("/summaries.csv")

metric = GEval(
    name="Action Item Quality",
    criteria="""
        Evaluate the action items extracted from the meeting transcript:
        1. Are all action items actually mentioned in the transcript?
        2. Are the descriptions clear and actionable?
        3. Are assignees only real speakers from the transcript (not invented names)?
        4. Are any important action items missed?
    """,
    evaluation_params=[
        LLMTestCaseParams.INPUT,
        LLMTestCaseParams.ACTUAL_OUTPUT
    ],
    model=OllamaModel(model="llama3.1"),
    threshold=0.5
)

geval_results = []

for i, row in df.head(100).iterrows():
    try:
        parsed_content = json.loads(row["generated_response"])
        if isinstance(parsed_content, dict):
            # Action items kısmını değerlendiriyoruz
            action_items = parsed_content.get("action_items", [])
            actual_output = json.dumps(action_items, ensure_ascii=False)
        else:
            actual_output = str(parsed_content)
    except (json.JSONDecodeError, TypeError):
        actual_output = str(row["generated_response"])

    test_case = LLMTestCase(
        input=row["transcript"],
        actual_output=actual_output
    )

    try:
        metric.measure(test_case)
        geval_results.append({
            "index": i,
            "score": metric.score,
            "reason": metric.reason,
            "passed": metric.is_successful()
        })
        print(f"[{i+1}/100] Score: {metric.score:.2f} | {metric.reason}")
    except Exception as e:
        geval_results.append({"index": i, "score": None, "reason": str(e), "passed": False})
        print(f"[{i+1}/100] HATA: {e}")

    time.sleep(3)

geval_df = pd.DataFrame(geval_results)
geval_df.to_csv("geval_results.csv", index=False)
print(f"\nOrtalama Skor: {geval_df['score'].mean():.2f}")
print(f"Başarılı: {geval_df['passed'].sum()}/100")

Output()

[1/100] Score: 0.10 | The transcript is a detailed account of the city council meeting where Council Bill 161 was debated and voted on. It provides insight into the decision-making process, various perspectives, and concerns raised during the discussion.


Output()

[2/100] Score: 0.00 | The council discussed and debated Council Bill 161, but there is no clear action item or decision made by the council regarding Council Bill 161.


Output()

...

[98/100] Score: 0.00 | The output is a summary of the discussion in a city council meeting, but it does not include all the details from the original text. The actual output should be a more detailed summary or a direct quote from the original text.


Output()

[99/100] Score: 0.00 | The transcript appears to be a summary of a city council meeting where a rezoning bill is being voted on. The debate revolves around the proposed zoning change, with some council members supporting it and others opposing it.


Output()

[100/100] Score: 0.40 | The action items are incomplete, as some decisions were made without assigning specific tasks or responsibilities. Additionally, the assignees for some action items are not clearly identified.

Ortalama Skor: 0.25
Başarılı: 13/100


In [18]:
import json
import pandas as pd
from rouge_score import rouge_scorer

df = pd.read_csv("/summaries.csv")
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

rouge_results = []

for i, row in df.head(100).iterrows():
    try:
        parsed_content = json.loads(row["generated_response"])
        if isinstance(parsed_content, dict):
            generated_summary = parsed_content.get("summary", str(parsed_content))
        else:
            generated_summary = str(parsed_content)
    except (json.JSONDecodeError, TypeError):
        generated_summary = str(row["generated_response"])

    reference_summary = str(row["summary"])  # ground truth — MeetingBank'ta zaten var

    scores = scorer.score(target=reference_summary, prediction=generated_summary)

    rouge_results.append({
        "index": i,
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
    })
    print(f"[{i+1}/100] ROUGE-1: {scores['rouge1'].fmeasure:.3f} | ROUGE-L: {scores['rougeL'].fmeasure:.3f}")

rouge_df = pd.DataFrame(rouge_results)
rouge_df.to_csv("rouge_results.csv", index=False)
print(f"\nOrtalama ROUGE-1: {rouge_df['rouge1'].mean():.3f}")
print(f"Ortalama ROUGE-2: {rouge_df['rouge2'].mean():.3f}")
print(f"Ortalama ROUGE-L: {rouge_df['rougeL'].mean():.3f}")

[1/100] ROUGE-1: 0.283 | ROUGE-L: 0.146
[2/100] ROUGE-1: 0.295 | ROUGE-L: 0.145
[3/100] ROUGE-1: 0.154 | ROUGE-L: 0.108
[4/100] ROUGE-1: 0.125 | ROUGE-L: 0.094
[5/100] ROUGE-1: 0.396 | ROUGE-L: 0.170
[6/100] ROUGE-1: 0.254 | ROUGE-L: 0.131
[7/100] ROUGE-1: 0.200 | ROUGE-L: 0.119
[8/100] ROUGE-1: 0.286 | ROUGE-L: 0.190
[9/100] ROUGE-1: 0.232 | ROUGE-L: 0.166
[10/100] ROUGE-1: 0.297 | ROUGE-L: 0.188
[11/100] ROUGE-1: 0.208 | ROUGE-L: 0.119
[12/100] ROUGE-1: 0.250 | ROUGE-L: 0.159
[13/100] ROUGE-1: 0.245 | ROUGE-L: 0.138
[14/100] ROUGE-1: 0.152 | ROUGE-L: 0.089
[15/100] ROUGE-1: 0.302 | ROUGE-L: 0.182
[16/100] ROUGE-1: 0.270 | ROUGE-L: 0.141
[17/100] ROUGE-1: 0.206 | ROUGE-L: 0.103
[18/100] ROUGE-1: 0.313 | ROUGE-L: 0.171
[19/100] ROUGE-1: 0.328 | ROUGE-L: 0.219
[20/100] ROUGE-1: 0.266 | ROUGE-L: 0.156
[21/100] ROUGE-1: 0.187 | ROUGE-L: 0.117
[22/100] ROUGE-1: 0.195 | ROUGE-L: 0.124
[23/100] ROUGE-1: 0.222 | ROUGE-L: 0.148
[24/100] ROUGE-1: 0.187 | ROUGE-L: 0.108
[25/100] ROUGE-1: 0.214 |